In [1]:
import pandas as pd

df = pd.read_excel('Data_Model_IoTMLCQ_2024.xlsx')
print(df.shape)
print(df.columns.tolist())
print(df.head())
print(df.info())

(43498, 24)
['Month', 'Average Fish Weight (g)', 'Survival Rate (%)', 'Disease Occurrence (Cases)', 'Temperature (°C)', 'Dissolved Oxygen (mg/L)', 'pH', 'Turbidity (NTU)', 'Month_Num', 'month_x', 'Oxygenation Interventions', 'Corrective Interventions', 'Average Temperature (°C)', 'High Temperature (°C)', 'Low Temperature (°C)', 'Precipitation (inches)', 'month_y', 'day', 'hour', 'oxigeno_scaled', 'ph', 'turbidez', 'Oxygenation Automatic', 'Corrective Measures']
     Month  Average Fish Weight (g)  Survival Rate (%)  \
0  January                   275.82              95.27   
1  January                   275.82              95.27   
2  January                   275.82              95.27   
3  January                   275.82              95.27   
4  January                   275.82              95.27   

   Disease Occurrence (Cases)  Temperature (°C)  Dissolved Oxygen (mg/L)  \
0                           2             27.47                     6.34   
1                           2    

In [2]:
# Create binary disease risk target
df['disease_risk'] = (df['Disease Occurrence (Cases)'] > 0).astype(int)
print(df['disease_risk'].value_counts())

# Select features
features = ['Temperature (°C)', 'Dissolved Oxygen (mg/L)', 'pH', 'Turbidity (NTU)']
df_clean = df[features + ['disease_risk']].dropna()

print("\nCleaned shape:", df_clean.shape)
print(df_clean.head())

disease_risk
1    43498
Name: count, dtype: int64

Cleaned shape: (43498, 5)
   Temperature (°C)  Dissolved Oxygen (mg/L)    pH  Turbidity (NTU)  \
0             27.47                     6.34  7.98              3.3   
1             27.47                     6.34  7.98              3.3   
2             27.47                     6.34  7.98              3.3   
3             27.47                     6.34  7.98              3.3   
4             27.47                     6.34  7.98              3.3   

   disease_risk  
0             1  
1             1  
2             1  
3             1  
4             1  


In [3]:
print(df['Disease Occurrence (Cases)'].describe())
print("\nUnique values:")
print(df['Disease Occurrence (Cases)'].value_counts().sort_index())

count    43498.000000
mean         1.142857
std          0.349931
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          2.000000
Name: Disease Occurrence (Cases), dtype: float64

Unique values:
Disease Occurrence (Cases)
1    37284
2     6214
Name: count, dtype: int64


In [4]:
df['disease_risk'] = (df['Disease Occurrence (Cases)'] == 2).astype(int)
print(df['disease_risk'].value_counts())
print("Positive rate:", df['disease_risk'].mean())

disease_risk
0    37284
1     6214
Name: count, dtype: int64
Positive rate: 0.14285714285714285


In [5]:
features = ['Temperature (°C)', 'Dissolved Oxygen (mg/L)', 'pH', 'Turbidity (NTU)']
df_clean = df[features + ['disease_risk']].dropna()

print("Cleaned shape:", df_clean.shape)
print(df_clean['disease_risk'].value_counts())

Cleaned shape: (43498, 5)
disease_risk
0    37284
1     6214
Name: count, dtype: int64


In [6]:
print(df_clean.shape)
print(df_clean['disease_risk'].value_counts())
print(df_clean.head())

(43498, 5)
disease_risk
0    37284
1     6214
Name: count, dtype: int64
   Temperature (°C)  Dissolved Oxygen (mg/L)    pH  Turbidity (NTU)  \
0             27.47                     6.34  7.98              3.3   
1             27.47                     6.34  7.98              3.3   
2             27.47                     6.34  7.98              3.3   
3             27.47                     6.34  7.98              3.3   
4             27.47                     6.34  7.98              3.3   

   disease_risk  
0             1  
1             1  
2             1  
3             1  
4             1  


In [7]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout

# Features and target
features = ['Temperature (°C)', 'Dissolved Oxygen (mg/L)', 'pH', 'Turbidity (NTU)']
X = df_clean[features].values
y = df_clean['disease_risk'].values

# Scale features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# ---------- Random Forest ----------
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train_rf, y_train_rf)

rf_pred = rf.predict(X_test_rf)
rf_prob = rf.predict_proba(X_test_rf)[:, 1]

print("=== Random Forest ===")
print(classification_report(y_test_rf, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test_rf, rf_prob))

# ---------- Create sequences for GRU / LSTM ----------
SEQ_LEN = 12   # last 12 readings (about 3 days if every 6 hours)
X_seq, y_seq = [], []

for i in range(len(X_scaled) - SEQ_LEN):
    X_seq.append(X_scaled[i:i+SEQ_LEN])
    y_seq.append(y[i+SEQ_LEN])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq
)

# ---------- GRU ----------
gru_model = Sequential([
    GRU(64, input_shape=(SEQ_LEN, len(features))),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
gru_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'AUC'])
gru_model.fit(X_train, y_train, epochs=15, batch_size=64, validation_split=0.1, verbose=1)

gru_prob = gru_model.predict(X_test).flatten()
gru_pred = (gru_prob > 0.5).astype(int)
print("\n=== GRU ===")
print(classification_report(y_test, gru_pred))
print("ROC-AUC:", roc_auc_score(y_test, gru_prob))

# ---------- LSTM ----------
lstm_model = Sequential([
    LSTM(64, input_shape=(SEQ_LEN, len(features))),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'AUC'])
lstm_model.fit(X_train, y_train, epochs=15, batch_size=64, validation_split=0.1, verbose=1)

lstm_prob = lstm_model.predict(X_test).flatten()
lstm_pred = (lstm_prob > 0.5).astype(int)
print("\n=== LSTM ===")
print(classification_report(y_test, lstm_pred))
print("ROC-AUC:", roc_auc_score(y_test, lstm_prob))

=== Random Forest ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7457
           1       1.00      1.00      1.00      1243

    accuracy                           1.00      8700
   macro avg       1.00      1.00      1.00      8700
weighted avg       1.00      1.00      1.00      8700

ROC-AUC: 1.0
Epoch 1/15


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


490/490 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - AUC: 0.9951 - accuracy: 0.9818 - loss: 0.0526 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 4.7188e-05
Epoch 2/15
490/490 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - AUC: 1.0000 - accuracy: 0.9998 - loss: 0.0012 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 1.3281e-04
Epoch 3/15
490/490 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - AUC: 0.9998 - accuracy: 0.9986 - loss: 0.0057 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 1.6400e-04
Epoch 4/15
490/490 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - AUC: 1.0000 - accuracy: 0.9998 - loss: 7.4441e-04 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 3.6260e-05
Epoch 5/15
490/490 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - AUC: 1.0000 - accuracy: 0.9998 - loss: 7.2009e-04 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 8.1091e-05
Epoch 6/15
490/490 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - AUC: 1.0000 - accuracy: 0.9999 - loss: 6.3046e-04 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 4.8445e-05
Epoch 7

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


490/490 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - AUC: 0.9967 - accuracy: 0.9838 - loss: 0.0453 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 1.2299e-04
Epoch 2/15
490/490 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - AUC: 0.9999 - accuracy: 0.9997 - loss: 0.0017 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 6.5142e-05
Epoch 3/15
490/490 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - AUC: 1.0000 - accuracy: 0.9998 - loss: 0.0013 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 1.2874e-05
Epoch 4/15
490/490 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - AUC: 0.9958 - accuracy: 0.9762 - loss: 0.0670 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 2.5540e-04
Epoch 5/15
490/490 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - AUC: 0.9999 - accuracy: 0.9997 - loss: 0.0014 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 1.7323e-04
Epoch 6/15
490/490 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - AUC: 0.9999 - accuracy: 0.9998 - loss: 0.0014 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 2.1756e-04
Epoch 7/15
490/490 ━━━

In [8]:
import joblib

# Save Random Forest
joblib.dump(rf, 'disease_risk_rf_model.pkl')
print("Random Forest saved")

Random Forest saved


In [9]:
# Save GRU
gru_model.save('disease_risk_gru.keras')
print("GRU saved")

GRU saved


In [10]:
# Save LSTM
lstm_model.save('disease_risk_lstm.keras')
print("LSTM saved")

LSTM saved
